In [ ]:
import ifcopenshell
import ifcopenshell.geom
import ifcopenshell.util.shape

import numpy as np

In [ ]:
def get_line_geometry(model_name, ifc_type):

    model_name = "26_02_D_STR.ifc" # 26-01-D-STR.ifc
    # Load IFC
    model = ifcopenshell.open(model_name)
    settings = ifcopenshell.geom.settings()

    # Extract beams
    beams = model.by_type("IfcBeam")

    # Initialize arrays
    beam_centers = []
    beam_starts = []
    beam_ends = []

    for beam in beams:

        try:
            shape = ifcopenshell.geom.create_shape(settings, beam)

            # Global vertex coordinates
            vertices = np.array(ifcopenshell.util.shape.get_shape_vertices(shape, shape.geometry))

            # Find main beam direction using PCA
            center = vertices.mean(axis=0)
            U, S, v = np.linalg.svd(vertices - center)

            direction = v[0]
            projections = (vertices - center) @ direction

            # Find endpoints
            start = center + projections.min() * direction
            end = center + projections.max() * direction

            # Store points
            beam_centers.append(center)
            beam_starts.append(start)
            beam_ends.append(end)

        except Exception as e:
            print(f"Could not process beam {beam.Name}: {e}")


    # Convert to numpy arrays
    beam_centers = np.array(beam_centers)
    beam_starts = np.array(beam_starts)
    beam_ends = np.array(beam_ends)
    return beam_centers, beam_starts, beam_ends

In [ ]:
model_name()
beam_centers,get_line_geometry(model_name, ifc_type)